<a href="https://colab.research.google.com/github/bcalm-2/gen-ai-assignments-3.4.5/blob/main/Srashti_Assignment5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# TASK 1: load models

!pip install transformers sentence-transformers faiss-cpu -q

from transformers import pipeline
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

generator = pipeline("text2text-generation", model="google/flan-t5-base")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

print("ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 33.8 MB/s eta 0:00:00


In [ ]:
# TASK 2: data and faiss

documents = [
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks.",
    "NLP deals with text and language.",
    "Supervised learning uses labeled data.",
    "Unsupervised learning finds patterns."
]

doc_embeddings = embed_model.encode(documents)

dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings))

print("data stored")

In [ ]:
# TASK 3: rag query

def rag_query(query):
    q_emb = embed_model.encode([query])

    D, I = index.search(np.array(q_emb), k=2)
    context = " ".join([documents[i] for i in I[0]])

    prompt = f"""
    Answer using this context only.
    If not found say 'Not found'.

    Context: {context}
    Question: {query}
    Answer:
    """

    result = generator(prompt, max_length=100)
    return result[0]["generated_text"]

print(rag_query("What is machine learning?"))

In [ ]:
# TASK 4: evaluation

import pandas as pd

data = {
    "Query": ["What is ML?", "What is DL?"],
    "Accuracy": [5, 4],
    "Quality": [5, 4],
    "Grounding": [5, 5],
    "Hallucination": ["No", "No"]
}

df = pd.DataFrame(data)
print(df)

df.to_csv("evaluation.csv", index=False)